In [24]:
import os
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit
import optuna


pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

In [27]:
imf1_X_train_val = pd.read_csv('../data/processed/imf1_x_train_val.csv').to_numpy()
imf1_X_train_val

array([['2023-01-01 00:00:00+00:00', -2.2909973144732145,
        -4.417527280781499, ..., 2.083590550364991, 0.4532380176105384,
        -3.9236565409230786],
       ['2023-01-01 01:00:00+00:00', 1.150266450366112,
        -2.2909973144732145, ..., 4.356008199562555, 4.204114405239738,
        11.434201346597757],
       ['2023-01-01 02:00:00+00:00', 0.9152438987440884,
        1.150266450366112, ..., 4.962825606563697, -2.8580497242764267,
        -6.699999843616185],
       ...,
       ['2024-10-19 21:00:00+00:00', 1.1439394598900587,
        -0.0658700942357578, ..., -3.1459900119686974,
        5.0006515961422195, 0.0850950035071931],
       ['2024-10-19 22:00:00+00:00', 1.3255004450091608,
        1.1439394598900587, ..., 3.077215506071964, 0.9197531571941876,
        6.156177830654036],
       ['2024-10-19 23:00:00+00:00', -0.3245981870426655,
        1.3255004450091608, ..., 1.4331656900378056, -5.477063012137696,
        -4.028496302584365]], dtype=object)

In [ ]:
def objective(trial):
    hidden_size = trial.suggest_int('hidden_size', 32, 256, log=True)
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    dropout = trial.suggest_float('dropout', 0.0, 0.7)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128, 256])
    num_layers = trial.suggest_int('num_layers', 1, 4)

    fold_scores, step = [], 0
    for (train_idx, val_idx) in tscv.split(X_train_val):
        X_train, X_val = X_train_val.iloc[train_idx].to_numpy(), X_train_val.iloc[val_idx].to_numpy()
        y_train, y_val = y_train_val.iloc[train_idx].to_numpy(), y_train_val.iloc[val_idx].to_numpy()

        X_train, y_train = create_sequences(X_train, y_train, seq_len=24*7, pred_len=24)
        X_val, y_val = create_sequences(X_val, y_val, seq_len=24*7, pred_len=24)

        X_train = torch.tensor(X_train, dtype=torch.float32, device=device)
        y_train = torch.tensor(y_train, dtype=torch.float32, device=device)
        X_val = torch.tensor(X_val, dtype=torch.float32, device=device)
        y_val = torch.tensor(y_val, dtype=torch.float32, device=device)

        mod = GRUnn(
            input_size=X_train.shape[2],
            hidden_size=hidden_size,
            output_size=y_train.shape[2],
            num_layers=num_layers,
            dropout=dropout
        ).to(device)

        mod.init_norm(X_train, y_train)
        X_train_norm = (X_train - mod.x_min) / (mod.x_max - mod.x_min)
        y_train_norm = (y_train - mod.y_min) / (mod.y_max - mod.y_min)
        X_val_norm = (X_val - mod.x_min) / (mod.x_max - mod.x_min)
        y_val_norm = (y_val - mod.y_min) / (mod.y_max - mod.y_min)
        
        optimizer = torch.optim.Adam(mod.parameters(), lr=lr)
        criterion = nn.MSELoss()

        train_dataset = TensorDataset(X_train_norm, y_train_norm)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
        val_dataset = TensorDataset(X_val_norm, y_val_norm)
        val_loader = DataLoader(val_dataset, batch_size=len(val_dataset)//10, shuffle=False)

        best_val_loss = np.inf
        best_state = None

        for epoch in range(100):
            mod.train()
            for xb_train_norm, yb_train_norm in train_loader:
                optimizer.zero_grad()
                yb_train_norm_pred = mod(xb_train_norm)
                loss = criterion(yb_train_norm_pred, yb_train_norm)
                loss.backward()
                optimizer.step()
            
            mod.eval()
            with torch.no_grad():
                val_loss = []
                for xb_val_norm, yb_val_norm in val_loader:
                    yb_val_norm_pred = mod(xb_val_norm)
                    batch_loss = criterion(yb_val_norm_pred, yb_val_norm).item()
                    val_loss.append(batch_loss)
                val_loss = np.mean(val_loss)

            trial.report(val_loss, step=step)
            if trial.should_prune():
                raise optuna.TrialPruned()
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = mod.state_dict().copy()

            step += 1

        if best_state:
            mod.load_state_dict(best_state)
        mod.eval()
        with torch.no_grad():
            curr_score = []
            for xb_val_norm, yb_val_norm in val_loader:
                yb_val_norm_pred = mod(xb_val_norm)
                yb_val_pred = mod.target_denorm(yb_val_norm_pred)
                yb_val = mod.target_denorm(yb_val_norm)
                curr_score.append(((yb_val_pred - yb_val) ** 2).mean().item())
            fold_scores.append(np.mean(curr_score))

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return np.mean(fold_scores)

    
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(multivariate=True, seed=1),
    pruner=optuna.pruners.HyperbandPruner(min_resource=100, max_resource=300),
    study_name='gru_opt_tpe_hyperband_gpu'
)
study.optimize(objective, n_trials=250, timeout=60*60*48-60)
best_trial = study.best_trial
joblib.dump(study, os.path.join(SCRATCH_PATH, 'gru_tpe_hyperband_gpu.pkl'))
joblib.dump(best_trial, os.path.join(SCRATCH_PATH, 'gru.pkl'))

datetime    0
dtype: int64